# 07 - Artist Metadata Enrichment

## Goal

Enrich confidently matched MusicBrainz artists with metadata, genres, and tags for use in concert recommendations.

## Tasks

- Load resolved artist mappings
- Fetch MusicBrainz metadata by MBID
- Cache API responses
- Extract artist metadata
- Build genre and tag datasets
- Measure metadata coverage
- Save processed enrichment datasets

In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

In [5]:
project_path = Path("..")
mapping_file = (
    project_path / "data" / "processed" / "musicbrainz" / "artist_mapping.csv"
)

artist_mapping = pd.read_csv(mapping_file)
artist_mapping.shape

(472, 19)

In [9]:
resolved_artists = (
    artist_mapping[artist_mapping["mbid"].notna()].copy()
)

print("Total artists:", len(artist_mapping))
print("Resolved artists:", len(resolved_artists))

Total artists: 472
Resolved artists: 305


In [11]:
musicbrainz_base_url = (
    "https://musicbrainz.org/ws/2/artist"
)
headers = {
    "User-Agent": "GigRouteEurope/1.0"
}

In [13]:
metadata_cache_path = (
    project_path / "data" / "raw" / "musicbrainz" / "artist_metadata_cache.json"
)

metadata_cache_path.parent.mkdir(
    parents= True,
    exist_ok= True
)

In [15]:
if metadata_cache_path.exists():
    with open(
        metadata_cache_path,
        "r",
        encoding="utf-8"
    ) as file:
        metadata_cache = json.load(file)
else:
    metadata_cache = {}
print("Cached artist metadata:", len(metadata_cache))

Cached artist metadata: 0


In [16]:
def fetch_artist_metadata(
    mbid,
    session,
    max_retries=3
):
    url = f"{musicbrainz_base_url}/{mbid}"

    params = {
        "fmt": "json",
        "inc": "genres+tags"
    }

    for attempt in range(1, max_retries + 1):

        try:
            response = session.get(
                url,
                params=params,
                headers=headers,
                timeout=45
            )

            if response.status_code == 200:
                return {
                    "status_code": 200,
                    "data": response.json()
                }

            if response.status_code in [
                429,
                502,
                503,
                504
            ]:
                wait_seconds = attempt * 5

                print(
                    f"Temporary API error "
                    f"{response.status_code}. "
                    f"Retrying in {wait_seconds}s..."
                )

                time.sleep(wait_seconds)
                continue

            return {
                "status_code": response.status_code,
                "data": None
            }

        except requests.exceptions.RequestException:

            wait_seconds = attempt * 5

            print(
                f"Request failed for {mbid}. "
                f"Attempt {attempt}/{max_retries}. "
                f"Retrying in {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

    return {
        "status_code": None,
        "data": None
    }

In [17]:
mbids = (
    resolved_artists["mbid"]
    .dropna()
    .unique()
)

print("Unique MBIDs:", len(mbids))

Unique MBIDs: 305


In [18]:
session = requests.Session()

for index, mbid in enumerate(
    mbids,
    start=1
):

    cached_result = metadata_cache.get(mbid)

    if (
        cached_result is not None
        and cached_result.get("status_code") == 200
    ):
        print(
            f"{index}/{len(mbids)} "
            f"| Cached | {mbid}"
        )
        continue

    result = fetch_artist_metadata(
        mbid,
        session
    )

    metadata_cache[mbid] = result

    with open(
        metadata_cache_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            metadata_cache,
            file,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"{index}/{len(mbids)} "
        f"| Status {result['status_code']} "
        f"| {mbid}"
    )

    time.sleep(1.1)

1/305 | Status 200 | 430a7a48-14ba-4be0-8226-7340f9b3bbaf
2/305 | Status 200 | 7de398f0-b4cf-476d-ac60-41d47d3511ca
3/305 | Status 200 | 16456fed-c9f2-4adf-b6ea-97b648c474d2
4/305 | Status 200 | 1df15ee0-b52b-4315-9cb9-bc5a27a685e9
5/305 | Status 200 | 98a1e0ab-35fa-40dd-b62c-9fda46fdb061
Temporary API error 503. Retrying in 5s...
6/305 | Status 200 | 34356aec-8c4b-4f15-997c-e972cdede64d
Temporary API error 503. Retrying in 5s...
7/305 | Status 200 | 8ebd161e-f45e-41b9-8019-fcbd094c327f
8/305 | Status 200 | 4b877353-2b15-432c-a439-f06cb210e033
9/305 | Status 200 | 6dee6b6f-aa72-424c-9c8c-0d31ba5ff673
10/305 | Status 200 | 32d2897c-4c91-46db-8161-f51d3f4f0d93
Temporary API error 503. Retrying in 5s...
11/305 | Status 200 | 1fda852b-92e9-4562-82fa-c52820a77b23
12/305 | Status 200 | 07a17571-81fc-4cf8-a634-98f0d926d313
13/305 | Status 200 | 6ef3b33e-848c-4773-867b-5fb1397ef408
14/305 | Status 200 | 380429da-3827-43fd-9e67-558e4c6a91bf
Temporary API error 503. Retrying in 5s...
15/305 | St